In [ ]:
import pandas as pd
import numpy as np
import pickle
import datetime
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm

In [ ]:
with open("data/characteristics_with_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

numpc = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10] 
predictions = []
dates = []
y_test_list = []
dic_r2_all = {}
numpc_time = {}

tscv = TimeSeriesSplit(n_splits=5)
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    X_train_val = train_data.drop(['permno', 'DATE', 'RET'], axis=1)
    y_train_val = train_data['RET']
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values

    for train_index, val_index in tscv.split(X_train_val):
        X_train = X_train_val.iloc[train_index]
        y_train = y_train_val.iloc[train_index]
        
        X_val = X_train_val.iloc[val_index]
        y_val = y_train_val.iloc[val_index]
        
        mse = np.zeros((len(numpc), 1))
        
      
        for i in range(len(numpc)):
            pls_val = PLSRegression(n_components=numpc[i], scale=False)
            pls_val.fit(X_train, y_train)
            Yval_predict = pls_val.predict(X_val)
            Yval_predict = Yval_predict.ravel() 
            mse[i, 0] = np.sqrt(mean_squared_error(y_val, Yval_predict))
        
        
        optim_numpc = numpc[np.argmin(mse)]
        pls = PLSRegression(n_components=optim_numpc, scale=False)
        pls.fit(np.concatenate((X_train, X_val)), np.concatenate((y_train, y_val)))
        preds = pls.predict(X_test)
        preds = preds.ravel() 
        
       
        predictions.append(preds)
        dates.append(test_data['DATE'].values)
        y_test_list.append(y_test)
        
       
        r2 = 1 - sum((y_test - preds)**2) / sum(y_test**2)
        dic_r2_all[f"r2.{year}"] = r2  
        numpc_time[f"numpc.{year}"] = optim_numpc
predictions_all = np.concatenate(predictions, axis=0)
y_test_list_all = np.concatenate(y_test_list, axis=0)
dates_all = np.concatenate(dates, axis=0)
R2OOS_PLS = 1 - sum((y_test_list_all - predictions_all)**2) / sum(y_test_list_all**2)
print("R2OOS partial least squares: ", R2OOS_PLS)